# Testes de Hipóteses — Exemplos Passo a Passo

Este notebook reproduz em Python todos os exemplos do material de **Testes de Hipóteses**, incluindo as tabelas de cálculo de cada teste.

**Exemplos cobertos:**
1. Teste de Kolmogorov-Smirnov  
2. Teste de Shapiro-Wilk  
3. Teste de Shapiro-França  
4. Teste χ² de Bartlett  
5. Teste z para uma média populacional  
6. Teste t de Student — uma amostra  
7. Teste t de Student — duas amostras independentes  
8. Teste t de Student — duas amostras emparelhadas
9. Teste Qui-Quadrado de Homogeneidade

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)


---
## Exemplo 1 — Teste de Kolmogorov-Smirnov (n=36)

- **H₀:** Dados ~ N(µ,σ)  
- **H₁:** Dados ≁ N(µ,σ)  
- **α = 5%**


In [ ]:
# ==============================
# DADOS
# ==============================
dados = np.array([
    52, 50, 36, 40, 30, 42, 38, 38, 52, 44, 36, 34,
    50, 42, 34, 55, 36, 55, 42, 52, 34, 48, 55, 44,
    44, 30, 48, 40, 40, 44, 40, 44, 38, 36, 50, 42
])

n = len(dados)
media = dados.mean()
dp = dados.std(ddof=1)

# ==============================
# CRIAR TABELA KS — Frequência Observada vs Esperada
# ==============================
valores = np.sort(np.unique(dados))
rows = []
Fobs_ant = 0.0

for xi in valores:
    fi = int(np.sum(dados == xi))
    fac = int(np.sum(dados <= xi))
    Fobs = fac / n
    zi = (xi - media) / dp
    Fesp = stats.norm.cdf(zi)
    d1 = abs(Fesp - Fobs)
    d2 = abs(Fesp - Fobs_ant)

    rows.append({
        'Xi': xi,
        'fi': fi,
        'F_obs(Xi)': F"{Fobs:.4f}",
        'Zi': F"{zi:.2f}",
        'F_esp(Xi)': F"{Fesp:.4f}",
        '|F_esp - F_obs|': F"{d1:.4f}",
        '|F_esp - F_obs(i-1)|': F"{d2:.4f}",
    })
    Fobs_ant = Fobs

tab = pd.DataFrame(rows)

# ==============================
# BLOCO DE RESULTADOS
# ==============================
Dcalc = max(tab['|F_esp - F_obs|'].astype(float).max(),
            tab['|F_esp - F_obs(i-1)|'].astype(float).max())
Dc = 0.23
_, pval = stats.kstest(dados, 'norm', args=(media, dp))
decisao = "REJEITAR H₀" if Dcalc > Dc else "NÃO REJEITAR H₀ — dados compatíveis com Normal"

# ==============================
# PRINTAR TABELA
# ==============================
print("Tabela KS — Frequência Acumulada: Observada vs Esperada (Normal)\n")
display(tab)  # no notebook, display() deixa a tabela formatada

Tabela KS — Frequência Acumulada: Observada vs Esperada (Normal)



,Xi,fi,F_obs(Xi),Zi,F_esp(Xi),|F_esp - F_obs|,|F_esp - F_obs(i-1)|
0,30,2,0.0556,-1.78,0.0375,0.0180,0.0375
1,34,3,0.1389,-1.22,0.1118,0.0270,0.0563
2,36,4,0.2500,-0.94,0.1749,0.0751,0.0360
3,38,3,0.3333,-0.65,0.2568,0.0766,0.0068
4,40,4,0.4444,-0.37,0.3551,0.0894,0.0217
5,42,4,0.5556,-0.09,0.4641,0.0914,0.0197
6,44,5,0.6944,0.19,0.5760,0.1184,0.0205
7,48,2,0.7500,0.76,0.7749,0.0249,0.0805
8,50,3,0.8333,1.04,0.8501,0.0167,0.1001
9,52,3,0.9167,1.32,0.9063,0.0103,0.0730


In [ ]:
# ==============================
# RESULTADOS DO TESTE E ESTATÍSTICAS DESCRITIVAS
# ==============================
from scipy import stats

# Estatísticas descritivas
media = dados.mean()
erro_padrao = stats.sem(dados)  # erro padrão da média
desvio_padrao = dados.std(ddof=1)
variancia = dados.var(ddof=1)
minimo = dados.min()
maximo = dados.max()
soma = dados.sum()
contagem = len(dados)

print("\nEstatísticas Descritivas:")
print(f"  Média                = {media:.5f}")
print(f"  Erro padrão da média = {erro_padrao:.5f}")
print(f"  Desvio padrão        = {desvio_padrao:.5f}")
print(f"  Variância da amostra = {variancia:.5f}")
print(f"  Mínimo               = {minimo}")
print(f"  Máximo               = {maximo}")
print(f"  Soma                 = {soma}")
print(f"  Contagem             = {contagem}")

print("\nResultados do Teste KS:")
print(f"  D_calc   = {Dcalc:.4f}")
print(f"  D_crítico= {Dc}")
print(f"  P-valor  = {pval:.4f}")
print(f"  Decisão  = {decisao}")



Estatísticas Descritivas:
  Média                = 42.63889
  Erro padrão da média = 1.18332
  Desvio padrão        = 7.09991
  Variância da amostra = 50.40873
  Mínimo               = 30
  Máximo               = 55
  Soma                 = 1535
  Contagem             = 36

Resultados do Teste KS:
  D_calc   = 0.1184
  D_crítico= 0.23
  P-valor  = 0.6503
  Decisão  = NÃO REJEITAR H₀ — dados compatíveis com Normal


---
## Exemplo 2 — Teste de Shapiro-Wilk (n=24)

- **H₀:** Dados ~ N(µ,σ)  
- **H₁:** Dados ≁ N(µ,σ)  
- **α = 1%**


In [ ]:
# ==============================
# DADOS
# ==============================
dados_sw = np.array([
    15, 19, 23, 24, 28, 30, 32, 36,
    16, 20, 23, 25, 29, 31, 34, 39,
    18, 22, 24, 28, 30, 32, 36, 46
])
n_sw = len(dados_sw)
media_sw = dados_sw.mean()
dp_sw = dados_sw.std(ddof=1)
x_ord = np.sort(dados_sw)

# ==============================
# TABELA: Pares ordenados e contribuições para b
# ==============================
a_coef = np.array([0.4493, 0.3098, 0.2554, 0.2145, 0.1807, 0.1512,
                   0.1245, 0.0997, 0.0764, 0.0539, 0.0321, 0.0107])

rows = []
b = 0.0
for i, ai in enumerate(a_coef, start=1):
    x_sup = x_ord[n_sw - i]
    x_inf = x_ord[i - 1]
    contrib = ai * (x_sup - x_inf)
    b += contrib
    rows.append({
        'i': i,
        'X(i)': x_inf,
        f'X({n_sw}-i+1)': x_sup,
        'Diferença': x_sup - x_inf,
        'a_i': ai,
        'a_i × Diferença': round(contrib, 4)
    })

# Criar DataFrame
tab_sw = pd.DataFrame(rows)

# Exibir tabela de forma clara no notebook
print("Tabela Shapiro-Wilk — Pares ordenados e contribuições para b")
display(tab_sw)

Tabela Shapiro-Wilk — Pares ordenados e contribuições para b


,i,X(i),X(24-i+1),Diferença,a_i,a_i × Diferença
0,1,15,46,31,0.4493,13.9283
1,2,16,39,23,0.3098,7.1254
2,3,18,36,18,0.2554,4.5972
3,4,19,36,17,0.2145,3.6465
4,5,20,34,14,0.1807,2.5298
5,6,22,32,10,0.1512,1.5120
6,7,23,32,9,0.1245,1.1205
7,8,23,31,8,0.0997,0.7976
8,9,24,30,6,0.0764,0.4584
9,10,24,30,6,0.0539,0.3234


In [ ]:
[# ==============================
# ESTATÍSTICAS DESCRITIVAS
# ==============================
n_sw = len(dados_sw)
media_sw = dados_sw.mean()                     # Média
erro_padrao = stats.sem(dados_sw)              # Erro padrão da média
mediana = np.median(dados_sw)                  # Mediana
modo = stats.mode(dados_sw, keepdims=True).mode[0]  # Modo
dp = dados_sw.std(ddof=1)                      # Desvio padrão amostral
variancia = dados_sw.var(ddof=1)               # Variância amostral
curtose = stats.kurtosis(dados_sw, fisher=True)   # Curtose
assimetria = stats.skew(dados_sw)              # Assimetria
intervalo = dados_sw.max() - dados_sw.min()    # Amplitude
minimo = dados_sw.min()                        # Valor mínimo
maximo = dados_sw.max()                        # Valor máximo
soma = dados_sw.sum()                          # Soma
contagem = n_sw                                # Contagem

# ==============================
# CÁLCULO DO W (Shapiro-Wilk)
# ==============================
# Reutilizando 'b' calculado anteriormente nos pares simétricos
SQ = np.sum((dados_sw - media_sw)**2)
Wcalc = b**2 / SQ
Wc = 0.884  # valor crítico (exemplo)
stat_w, pval_w = stats.shapiro(dados_sw)
decisao = "REJEITAR H₀" if Wcalc < Wc else "NÃO REJEITAR H₀ — dados compatíveis com Normal"

# ==============================
# EXIBIÇÃO DOS RESULTADOS
# ==============================
print("Estatísticas Descritivas:\n")
print(f"  Média       = {media_sw:.4f}")
print(f"  Erro padrão = {erro_padrao:.6f}")
print(f"  Mediana     = {mediana}")
print(f"  Modo        = {modo}")
print(f"  Desvio padrão = {dp:.6f}")
print(f"  Variância da amostra = {variancia:.6f}")
print(f"  Curtose     = {curtose:.5f}")
print(f"  Assimetria  = {assimetria:.6f}")
print(f"  Intervalo   = {intervalo}")
print(f"  Mínimo      = {minimo}")
print(f"  Máximo      = {maximo}")
print(f"  Soma        = {soma}")
print(f"  Contagem    = {contagem}")

print("\nResultados do Teste Shapiro-Wilk:\n")
print(f"  b         = {b:.4f}")
print(f"  b²        = {b**2:.4f}")
print(f"  Σ(Xi-X̄)² = {SQ:.4f}")
print(f"  W_calc    = {Wcalc:.4f}")
print(f"  W_crítico = {Wc}")
print(f"  Estatística do SciPy = {stat_w:.4f}")
print(f"  P-valor   = {pval_w:.4f}")
print(f"  Decisão   = {decisao}")

Estatísticas Descritivas:

  Média       = 27.5000
  Erro padrão = 1.556892
  Mediana     = 28.0
  Modo        = 23
  Desvio padrão = 7.627183
  Variância da amostra = 58.173913
  Curtose     = -0.20604
  Assimetria  = 0.380169
  Intervalo   = 31
  Mínimo      = 15
  Máximo      = 46
  Soma        = 660
  Contagem    = 24

Resultados do Teste Shapiro-Wilk:

  b         = 36.1675
  b²        = 1308.0881
  Σ(Xi-X̄)² = 1338.0000
  W_calc    = 0.9776
  W_crítico = 0.884
  Estatística do SciPy = 0.9780
  P-valor   = 0.8565
  Decisão   = NÃO REJEITAR H₀ — dados compatíveis com Normal


---
## Exemplo 3 — Teste de Shapiro-França (n=60)

- **H₀:** Dados ~ N(µ,σ)  
- **H₁:** Dados ≁ N(µ,σ)  
- **α = 5%**


In [ ]:
dados_sf = np.array([
    85, 88, 66, 80, 80, 54, 91, 67, 80, 63, 76, 68,
    70, 80, 60, 55, 64, 59, 57, 70, 81, 59, 81, 76,
    74, 91, 72, 54, 60, 78, 59, 76, 70, 60, 79, 71,
    49, 57, 81, 93, 63, 73, 64, 78, 77, 61, 78, 72,
    67, 63, 73, 77, 67, 84, 68, 75, 65, 74, 60, 84
])
n_sf = len(dados_sf)
media_sf = dados_sf.mean()
dp_sf = dados_sf.std(ddof=1)
x_ord = np.sort(dados_sf)

# Tabela: X(i) observado vs valor esperado da Normal (mi)
mi = np.array([stats.norm.ppf(i/(n_sf+1)) for i in range(1, n_sf+1)])
x_esp = media_sf + dp_sf * mi  # valor esperado se fosse Normal perfeita

rows = []
for i in range(n_sf):
    rows.append({
        'i': i+1, 'X_obs(i)': x_ord[i],
        'X_esp(i) [Normal]': round(x_esp[i], 2),
        'Desvio (obs-esp)': round(x_ord[i] - x_esp[i], 2),
        'mi': round(mi[i], 4), 'mi·X(i)': round(mi[i]*x_ord[i], 2),
    })

tab_sf = pd.DataFrame(rows)
print("Tabela Shapiro-França — X observado vs X esperado (Normal)")
display(tab_sf)

num = np.sum(mi * x_ord)**2
den = np.sum(mi**2) * np.sum((dados_sf - media_sf)**2)
Wcalc_sf = num / den
stat_sf, pval_sf = stats.shapiro(dados_sf)

print(f"\n  W'_calc = {Wcalc_sf:.4f}   |   P-valor (Shapiro) = {pval_sf:.4f}")
print(f"  Decisão: {'REJEITAR H₀' if pval_sf < 0.05 else 'NÃO REJEITAR H₀ — dados compatíveis com Normal'}")


Tabela Shapiro-França — X observado vs X esperado (Normal)


,i,X_obs(i),X_esp(i) [Normal],Desvio (obs-esp),mi,mi·X(i)
0,1,49,48.9300,0.0700,-2.1347,-104.6000
1,2,54,51.9500,2.0500,-1.8413,-99.4300
2,3,54,53.9000,0.1000,-1.6529,-89.2500
3,4,55,55.3800,-0.3800,-1.5096,-83.0300
4,5,57,56.5900,0.4100,-1.3920,-79.3400
5,6,57,57.6300,-0.6300,-1.2909,-73.5800
6,7,59,58.5500,0.4500,-1.2016,-70.9000
7,8,59,59.3900,-0.3900,-1.1210,-66.1400
8,9,59,60.1500,-1.1500,-1.0470,-61.7800
9,10,60,60.8600,-0.8600,-0.9784,-58.7000



  W'_calc = 0.9888   |   P-valor (Shapiro) = 0.5232
  Decisão: NÃO REJEITAR H₀ — dados compatíveis com Normal


---
## Exemplo 4 — Teste χ² de Bartlett

- **H₀:** σ₁² = σ₂² = σ₃²  
- **H₁:** Pelo menos uma variância difere  
- **α = 5%**


In [ ]:
loja1 = np.array([620, 630, 610, 650, 585, 590, 630, 644, 595, 603, 570, 605, 622, 578])
loja2 = np.array([710, 780, 810, 755, 699, 680, 710, 850, 844, 730, 645, 688, 718, 702])
loja3 = np.array([924, 695, 854, 802, 931, 924, 847, 800, 769, 863, 901, 888, 757, 712])

# Tabela: Variâncias observadas vs variância esperada (pooled)
grupos = [loja1, loja2, loja3]
k = 3
ni = np.array([len(g) for g in grupos])
Si2 = np.array([g.var(ddof=1) for g in grupos])
N = ni.sum()
Sp2 = np.sum((ni-1)*Si2) / (N-k)

tab_b = pd.DataFrame({
    'Grupo': ['Loja 1', 'Loja 2', 'Loja 3'],
    'n': ni,
    'Média': [g.mean() for g in grupos],
    'S² observada': np.round(Si2, 2),
    'S² esperada (pooled)': [round(Sp2, 2)]*3,
    'Razão S²_obs/S²_esp': np.round(Si2/Sp2, 4),
})
print("Tabela Bartlett — Variâncias Observadas vs Esperada (H₀: iguais)")
print(tab_b.to_string(index=False))

q = (N-k)*np.log(Sp2) - np.sum((ni-1)*np.log(Si2))
c = 1 + (1/(3*(k-1))) * (np.sum(1/(ni-1)) - 1/(N-k))
Bcalc = q / c
chi2_c = stats.chi2.ppf(0.95, df=k-1)
pval_b = 1 - stats.chi2.cdf(Bcalc, df=k-1)

print(f"\n  Sp² (esperada sob H₀) = {Sp2:.2f}")
print(f"  B_calc = {Bcalc:.4f}   |   χ²_crítico(gl=2) = {chi2_c:.4f}   |   P-valor = {pval_b:.4f}")
print(f"  Decisão: {'REJEITAR H₀ — variâncias heterogêneas' if Bcalc > chi2_c else 'NÃO REJEITAR H₀ — variâncias homogêneas'}")


Tabela Bartlett — Variâncias Observadas vs Esperada (H₀: iguais)
 Grupo  n    Média  S² observada  S² esperada (pooled)  Razão S²_obs/S²_esp
Loja 1 14 609.4286      595.6500             3565.9200               0.1670
Loja 2 14 737.2143     3874.6400             3565.9200               1.0866
Loja 3 14 833.3571     6227.4800             3565.9200               1.7464

  Sp² (esperada sob H₀) = 3565.92
  B_calc = 14.4426   |   χ²_crítico(gl=2) = 5.9915   |   P-valor = 0.0007
  Decisão: REJEITAR H₀ — variâncias heterogêneas


---
## Exemplo 5 — Teste z para Média Populacional

- **H₀:** µ = 4,2 g  
- **H₁:** µ < 4,2 g (unilateral esquerda)  
- **α = 5%**


In [ ]:
mu0 = 4.2; sigma = 1.0; n_z = 42; x_bar = 3.9; alpha = 0.05
se = sigma / np.sqrt(n_z)
Zcalc = (x_bar - mu0) / se
zc = stats.norm.ppf(alpha)
pval = stats.norm.cdf(Zcalc)

tab_z = pd.DataFrame({
    'Item': ['Média esperada (µ₀)', 'Média observada (X̄)', 'σ conhecido', 'n',
             'Erro Padrão (σ/√n)', 'Z_calc', 'Z_crítico (α=5%)', 'P-valor', 'Decisão'],
    'Valor': [mu0, x_bar, sigma, n_z, round(se, 4), round(Zcalc, 4),
              round(zc, 4), round(pval, 4),
              'REJEITAR H₀' if Zcalc < zc else 'NÃO REJEITAR H₀']
})
print("Tabela Teste z — Valor Esperado vs Observado")
print(tab_z.to_string(index=False))


Tabela Teste z — Valor Esperado vs Observado
                Item       Valor
 Média esperada (µ₀)      4.2000
Média observada (X̄)      3.9000
         σ conhecido      1.0000
                   n          42
  Erro Padrão (σ/√n)      0.1543
              Z_calc     -1.9442
    Z_crítico (α=5%)     -1.6449
             P-valor      0.0259
             Decisão REJEITAR H₀


---
## Exemplo 6 — Teste t de Student (uma amostra)

- **H₀:** µ = 18 min  
- **H₁:** µ < 18 min (unilateral esquerda)  
- **α = 1%**


In [ ]:
mu0_t = 18.0; n_t = 25; x_bar_t = 16.808; s_t = 2.733; gl = n_t - 1
se_t = s_t / np.sqrt(n_t)
Tcalc = (x_bar_t - mu0_t) / se_t
tc = stats.t.ppf(0.01, df=gl)
pval_t = stats.t.cdf(Tcalc, df=gl)

tab_t = pd.DataFrame({
    'Item': ['Média esperada (µ₀)', 'Média observada (X̄)', 'Desvio padrão (S)', 'n', 'gl',
             'Erro Padrão (S/√n)', 'T_calc', 't_crítico (α=1%, gl=24)', 'P-valor', 'Decisão'],
    'Valor': [mu0_t, x_bar_t, s_t, n_t, gl, round(se_t, 4), round(Tcalc, 4),
              round(tc, 4), round(pval_t, 4),
              'REJEITAR H₀ — tempo melhorou' if Tcalc < tc else 'NÃO REJEITAR H₀']
})
print("Tabela Teste t (1 amostra) — Valor Esperado vs Observado")
print(tab_t.to_string(index=False))


Tabela Teste t (1 amostra) — Valor Esperado vs Observado
                   Item           Valor
    Média esperada (µ₀)         18.0000
   Média observada (X̄)         16.8080
      Desvio padrão (S)          2.7330
                      n              25
                     gl              24
     Erro Padrão (S/√n)          0.5466
                 T_calc         -2.1808
t_crítico (α=1%, gl=24)         -2.4922
                P-valor          0.0196
                Decisão NÃO REJEITAR H₀


---
## Exemplo 7 — Teste t para Duas Amostras Independentes

- **H₀:** µ₁ = µ₂  
- **H₁:** µ₁ ≠ µ₂ (bilateral)  
- **α = 5%**


In [ ]:
forn1 = np.array([22.8,23.4,26.2,24.3,22.0,24.8,26.7,25.1,23.1,22.8,
                  25.6,25.1,24.3,24.2,22.8,23.2,24.7,26.5,24.5,23.6,
                  23.9,22.8,25.4,26.7,22.9,23.5,23.8,24.6,26.3,22.7])
forn2 = np.array([26.8,29.3,28.4,25.6,29.4,27.2,27.6,26.8,25.4,28.6,
                  29.7,27.2,27.9,28.4,26.0,26.8,27.5,28.5,27.3,29.1,
                  29.2,25.7,28.4,28.6,27.9,27.4,26.7,26.8,25.6,26.1])

n1, n2 = len(forn1), len(forn2)
xb1, xb2 = forn1.mean(), forn2.mean()
s1, s2 = forn1.std(ddof=1), forn2.std(ddof=1)

# Tabela comparativa
tab_comp = pd.DataFrame({
    'Estatística': ['n', 'Média (observada)', 'Desvio Padrão', 'Variância'],
    'Fornecedor 1': [n1, round(xb1, 4), round(s1, 4), round(s1**2, 4)],
    'Fornecedor 2': [n2, round(xb2, 4), round(s2, 4), round(s2**2, 4)],
    'Esperado (H₀: iguais)': ['—', round((xb1+xb2)/2, 4), '—', '—'],
})
print("Tabela Comparativa — Fornecedores")
print(tab_comp.to_string(index=False))

# Bartlett
stat_b, pval_b = stats.bartlett(forn1, forn2)
var_iguais = pval_b > 0.05

# Teste t (escolhe automaticamente)
if var_iguais:
    Sp = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2)/(n1+n2-2))
    Tcalc = (xb1-xb2)/(Sp*np.sqrt(1/n1+1/n2))
    gl = n1+n2-2
    tipo = "Pooled (var. iguais)"
else:
    Tcalc = (xb1-xb2)/np.sqrt(s1**2/n1 + s2**2/n2)
    gl = int((s1**2/n1+s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1)+(s2**2/n2)**2/(n2-1)))
    tipo = "Welch (var. diferentes)"

tc = stats.t.ppf(0.975, df=gl)
pval = 2*(1-stats.t.cdf(abs(Tcalc), df=gl))

print(f"\nBartlett: P-valor = {pval_b:.4f} → Variâncias {'iguais' if var_iguais else 'diferentes'}")
print(f"\n  Diferença observada (X̄1-X̄2) = {xb1-xb2:.4f}")
print(f"  Diferença esperada (H₀)      = 0")
print(f"  Tipo: {tipo}   |   gl = {gl}")
print(f"  T_calc = {Tcalc:.4f}   |   t_crítico = ±{tc:.4f}   |   P-valor = {pval:.6f}")
print(f"  Decisão: {'REJEITAR H₀ — médias diferentes' if abs(Tcalc) > tc else 'NÃO REJEITAR H₀'}")


Tabela Comparativa — Fornecedores
      Estatística  Fornecedor 1  Fornecedor 2 Esperado (H₀: iguais)
                n       30.0000       30.0000                     —
Média (observada)       24.2767       27.5300               25.9033
    Desvio Padrão        1.3454        1.2485                     —
        Variância        1.8101        1.5587                     —

Bartlett: P-valor = 0.6899 → Variâncias iguais

  Diferença observada (X̄1-X̄2) = -3.2533
  Diferença esperada (H₀)      = 0
  Tipo: Pooled (var. iguais)   |   gl = 58
  T_calc = -9.7084   |   t_crítico = ±2.0017   |   P-valor = 0.000000
  Decisão: REJEITAR H₀ — médias diferentes


---
## Exemplo 8 — Teste t para Amostras Emparelhadas

- **H₀:** µ_d = 0 (sem diferença)  
- **H₁:** µ_d ≠ 0 (bilateral)  
- **α = 5%**


In [ ]:
antes  = np.array([3.2, 3.6, 3.4, 3.8, 3.4, 3.5, 3.7, 3.2, 3.5, 3.9])
depois = np.array([3.0, 3.3, 3.5, 3.6, 3.4, 3.3, 3.4, 3.0, 3.2, 3.6])
n_e = len(antes)
di = antes - depois
d_bar = di.mean()
Sd = di.std(ddof=1)

# Tabela de diferenças: observado vs esperado (H₀: d=0)
tab_e = pd.DataFrame({
    'Operador': range(1, n_e+1),
    'Antes': antes,
    'Depois': depois,
    'di observado': di,
    'di esperado (H₀)': [0.0]*n_e,
    'Desvio (obs-esp)': di,
})
print("Tabela Emparelhado — Diferenças Observadas vs Esperadas")
print(tab_e.to_string(index=False))

se_d = Sd / np.sqrt(n_e)
Tcalc_e = d_bar / se_d
gl_e = n_e - 1
tc_e = stats.t.ppf(0.975, df=gl_e)
pval_e = 2*(1-stats.t.cdf(abs(Tcalc_e), df=gl_e))

print(f"\n  d̄ observado = {d_bar:.4f}   |   d̄ esperado (H₀) = 0")
print(f"  Sd = {Sd:.4f}   |   Erro Padrão = {se_d:.4f}")
print(f"  T_calc = {Tcalc_e:.4f}   |   t_crítico = ±{tc_e:.4f}   |   P-valor = {pval_e:.4f}")
print(f"  Decisão: {'REJEITAR H₀ — treinamento teve efeito' if abs(Tcalc_e) > tc_e else 'NÃO REJEITAR H₀'}")


Tabela Emparelhado — Diferenças Observadas vs Esperadas
 Operador  Antes  Depois  di observado  di esperado (H₀)  Desvio (obs-esp)
        1 3.2000  3.0000        0.2000            0.0000            0.2000
        2 3.6000  3.3000        0.3000            0.0000            0.3000
        3 3.4000  3.5000       -0.1000            0.0000           -0.1000
        4 3.8000  3.6000        0.2000            0.0000            0.2000
        5 3.4000  3.4000        0.0000            0.0000            0.0000
        6 3.5000  3.3000        0.2000            0.0000            0.2000
        7 3.7000  3.4000        0.3000            0.0000            0.3000
        8 3.2000  3.0000        0.2000            0.0000            0.2000
        9 3.5000  3.2000        0.3000            0.0000            0.3000
       10 3.9000  3.6000        0.3000            0.0000            0.3000

  d̄ observado = 0.1900   |   d̄ esperado (H₀) = 0
  Sd = 0.1370   |   Erro Padrão = 0.0433
  T_calc = 4.3846   |   t_

## Teste Qui-Quadrado de Homogeneidade — Estados do Nordeste

### Objetivo

O presente teste tem como objetivo verificar se a distribuição dos tipos de reprovação nos registros do SISAB é homogênea entre os estados da região Nordeste do Brasil. Em outras palavras, busca-se identificar se os estados apresentam padrões semelhantes ou distintos quanto às inconsistências registradas (como CBO, CNES, PROF e INE).

---

### Fundamentação

O teste qui-quadrado de homogeneidade é utilizado para comparar a distribuição de uma variável categórica entre diferentes grupos independentes. Neste contexto:

* **Variável categórica:** tipo de reprovação (CBO, CNES, PROF, INE)
* **Grupos:** estados da região Nordeste

A análise é realizada a partir de uma tabela de contingência contendo as frequências observadas de cada tipo de reprovação em cada estado.

---

### Hipóteses

* **Hipótese nula (H₀):**
  A distribuição dos tipos de reprovação é igual entre os estados do Nordeste.

* **Hipótese alternativa (H₁):**
  A distribuição dos tipos de reprovação difere entre os estados do Nordeste.

---

### Metodologia

1. Os dados mensais foram agregados para cada estado e tipo de reprovação.
2. Foi construída uma tabela de contingência com as frequências totais.
3. Aplicou-se o teste qui-quadrado de homogeneidade para avaliar diferenças entre os estados.
4. Foram verificadas as condições de validade do teste, especialmente as frequências esperadas.

---


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

Mounted at /content/drive


In [ ]:
pasta = '/content/drive/MyDrive/UFPB/CDIA/SISAB'

arquivos = os.listdir(pasta)

dfs = []

for arquivo in arquivos:
    if arquivo.endswith('.xlsx'):
        caminho = os.path.join(pasta, arquivo)

        df = pd.read_excel(caminho)

        df['MES'] = arquivo.replace('.xlsx', '')

        dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

df_final.head()

,Uf,Validação,Total,MES
0,PA,Reprovado(INE),2.486,11-25
1,PA,Reprovado(PROF),173.114,11-25
2,BA,Reprovado(PROF),154.215,11-25
3,PA,Reprovado(CNES+INE),52.000,11-25
4,AM,Reprovado(PROF),12.643,11-25


In [ ]:
df_final.columns = df_final.columns.str.strip().str.upper()

df_final['VALIDAÇÃO'] = df_final['VALIDAÇÃO'].str.strip().str.upper()
df_final['Uf'] = df_final['UF'].str.strip().str.upper()

df_final['Total'] = pd.to_numeric(df_final['TOTAL'], errors='coerce')

In [ ]:
nordeste = ['PB','PE','RN','CE','BA','AL','SE','PI','MA']

df_ne = df_final[df_final['UF'].isin(nordeste)]

In [ ]:
tabela_ne = df_ne.pivot_table(
    index='VALIDAÇÃO',
    columns='UF',
    values='TOTAL',
    aggfunc='sum'
).fillna(0)

tabela_ne = tabela_ne.loc[(tabela_ne.sum(axis=1) > 0), :]
tabela_ne = tabela_ne.loc[:, (tabela_ne.sum(axis=0) > 0)]

In [ ]:
# ================================
# TABELA DE DADOS OBSERVADOS
# ================================

tabela_observada = tabela_ne.copy()

# Adiciona coluna Total (soma por linha)
tabela_observada['Total'] = tabela_observada.sum(axis=1)
tabela_observada.loc['Total'] = tabela_observada.sum()
print("Tabela de Dados Observados (Contingência) com Total por linha:")
display(tabela_observada)

Tabela de Dados Observados (Contingência) com Total por linha:


UF,AL,BA,CE,MA,PB,PE,PI,RN,SE,Total
VALIDAÇÃO,,,,,,,,,,
REPROVADO(CBO),420.000,4121.998,6473.031,5540.000,1565.000,7173.182,4418.000,137.926,2232.986,32082.123
REPROVADO(CNES),136.000,617.892,2022.000,1708.790,846.000,1586.961,88.000,910.000,104.000,8019.643
REPROVADO(CNES+INE),344.000,4036.425,3366.000,1717.000,1512.000,2506.864,841.000,300.000,1260.000,15883.289
REPROVADO(CNES+INE+PROF),13.000,338.000,1542.000,117.000,121.000,1189.904,5.000,52.000,5.000,3382.904
REPROVADO(CNES+INE+PROF+CBO),5.000,166.000,23.000,55.000,13.000,360.000,2.000,2.000,8.000,634.000
REPROVADO(CNES+PROF),5.000,1046.000,190.000,67.000,4.000,640.000,45.000,102.000,91.000,2190.000
REPROVADO(CNES+PROF+CBO),9.000,200.000,159.000,504.000,4.000,12.000,43.000,61.000,8.000,1000.000
REPROVADO(INE),3258.286,78.141,964.128,528.790,2462.121,2253.243,3266.543,4078.799,2279.122,19169.173
REPROVADO(INE+CBO),34.000,0.000,0.000,0.000,4.000,0.000,0.000,0.000,0.000,38.000


In [ ]:
from scipy.stats import chi2_contingency

# Primeiro: roda o teste
chi2, p, dof, expected = chi2_contingency(tabela_ne)

print('Qui-quadrado:', chi2)
print('p-valor:', p)

# Depois: verifica as condições
total_celulas = expected.size
baixas = (expected < 5).sum()

print("Total de células:", total_celulas)
print("Células < 5:", baixas)
print("Percentual:", baixas / total_celulas)

Qui-quadrado: 58638.7418766041
p-valor: 0.0
Total de células: 117
Células < 5: 5
Percentual: 0.042735042735042736


In [ ]:
# ================================
# TABELA DE DADOS ESPERADOS
# ================================

# Cria a tabela de valores esperados com os mesmos índices e colunas da observada
tabela_esperada = pd.DataFrame(
    expected,
    index=tabela_ne.index,
    columns=tabela_ne.columns
)

# Adiciona coluna Total (soma por linha)
tabela_esperada['Total'] = tabela_esperada.sum(axis=1)

# Adiciona linha Total (soma por coluna original, sem a coluna Total)
tabela_esperada.loc['Total'] = tabela_esperada.iloc[:-1, :-1].sum()

# Corrige a célula Total / Total
tabela_esperada.loc['Total', 'Total'] = tabela_esperada['Total'].sum()

# Exibe a tabela final
print("Tabela de Dados Esperados")
display(tabela_esperada)

Tabela de Dados Esperados


UF,AL,BA,CE,MA,PB,PE,PI,RN,SE,Total
VALIDAÇÃO,,,,,,,,,,
REPROVADO(CBO),1387.609266,5159.623625,5104.946433,4701.217370,2534.113805,6823.935746,2942.389013,1812.663325,1615.624418,32082.123
REPROVADO(CNES),346.863920,1289.763134,1276.095348,1175.174254,633.458329,1705.795110,735.515834,453.115673,403.861398,8019.643
REPROVADO(CNES+INE),686.980690,2554.437971,2527.368264,2327.489179,1254.594714,3378.409327,1456.724514,897.417401,799.866940,15883.289
REPROVADO(CNES+INE+PROF),146.316655,544.057243,538.291799,495.720531,267.209989,719.550871,310.260626,191.136541,170.359745,3382.904
REPROVADO(CNES+INE+PROF+CBO),27.421635,101.963370,100.882851,92.904444,50.078611,134.853147,58.146857,35.821462,31.927622,634.000
REPROVADO(CNES+PROF),94.721421,352.207855,348.475464,320.915983,172.984476,465.817654,200.854287,123.736596,110.286264,2190.000
REPROVADO(CNES+PROF+CBO),43.251791,160.825505,159.121216,146.536979,78.988345,212.702125,91.714286,56.500729,50.359024,1000.000
REPROVADO(INE),829.101056,3082.891924,3050.222123,2808.992693,1514.141253,4077.323837,1758.087020,1083.072242,965.340852,19169.173
REPROVADO(INE+CBO),1.643568,6.111369,6.046606,5.568405,3.001557,8.082681,3.485143,2.147028,1.913643,38.000


In [ ]:
diferenca_relativa = tabela_observada.iloc[:-1, :-1] / tabela_esperada.iloc[:-1, :-1]
display(diferenca_relativa.style.background_gradient(cmap='coolwarm'))

UF,AL,BA,CE,MA,PB,PE,PI,RN,SE
VALIDAÇÃO,,,,,,,,,
REPROVADO(CBO),0.302679,0.798895,1.267992,1.178418,0.617573,1.051180,1.501501,0.076090,1.382119
REPROVADO(CNES),0.392085,0.479074,1.584521,1.454074,1.335526,0.930335,0.119644,2.008317,0.257514
REPROVADO(CNES+INE),0.500742,1.580162,1.331820,0.737705,1.205170,0.742025,0.577323,0.334293,1.575262
REPROVADO(CNES+INE+PROF),0.088848,0.621258,2.864617,0.236020,0.452827,1.653676,0.016115,0.272057,0.029350
REPROVADO(CNES+INE+PROF+CBO),0.182338,1.628036,0.227987,0.592006,0.259592,2.669571,0.034396,0.055832,0.250567
REPROVADO(CNES+PROF),0.052786,2.969837,0.545232,0.208777,0.023123,1.373928,0.224043,0.824332,0.825125
REPROVADO(CNES+PROF+CBO),0.208084,1.243584,0.999238,3.439405,0.050640,0.056417,0.468847,1.079632,0.158859
REPROVADO(INE),3.929902,0.025347,0.316085,0.188249,1.626084,0.552628,1.858010,3.765953,2.360951
REPROVADO(INE+CBO),20.686701,0.000000,0.000000,0.000000,1.332642,0.000000,0.000000,0.000000,0.000000


In [ ]:
alpha = 0.05

if p < alpha:
    print("Rejeitamos H0: há diferenças entre os estados.")
else:
    print("Não rejeitamos H0: não há evidência de diferença.")

Rejeitamos H0: há diferenças entre os estados.
